# Profiling vLLM inference (NVIDIA Nsight Systems)



Prototyping some experiments inside a Google Colab notebook

Check CUDA version and install compatible Nsight Systems package

In [ ]:
! nvcc --version

In [ ]:
! apt -q install nsight-systems-2025.3.2

Install vLLM and the usual suspects

In [ ]:
%pip install vllm accelerate torch --quiet
%pip install transformers huggingface_hub[cli] --quiet
%pip install openai --quiet

Enter Hugging Face token

In [ ]:
! hf auth login

Check vLLM version and install compatible benchmark scripts

In [ ]:
import os

os.environ['VLLM_VERSION'] = os.popen("pip show vllm | sed -n 's/^Version: //p'").read().strip()

! git clone --depth 1 --branch "v${VLLM_VERSION}" https://github.com/vllm-project/vllm.git

os.environ['VLLM_DIR'] = "/content/vllm"

Run benchmark script

In [ ]:
! nsys profile -o report.nsys-rep --trace-fork-before-exec=true \
                                 --cuda-graph-trace=node \
                              python $VLLM_DIR/benchmarks/benchmark_latency.py \
                                 --model NousResearch/Llama-2-7b-hf \
                                 --num-iters-warmup 5 \
                                 --num-iters 1 \
                                 --batch-size 16 \
                                 --input-len 512 \
                                 --output-len 8

Analyze the report of the profiler

In [ ]:
! nsys stats /content/report.nsys-rep

Start vLLM with a model small enough for Colab

In [ ]:
#os.environ['VLLM_LOGGING_LEVEL']='DEBUG'

! nohup python3 -m vllm.entrypoints.openai.api_server \
    --model NousResearch/Llama-2-7b-hf \
    --port 8000 \
    --max-model-len 752 > vllm_server.log 2>&1 &

Query the model

In [ ]:
from openai import OpenAI

# Use your local vLLM server
client = OpenAI(
    base_url = "http://127.0.0.1:8000/v1",
    api_key  = "EMPTY"  # not used, but required by the SDK
)

response = client.completions.create(
    model  = "NousResearch/Llama-2-7b-hf",
    prompt = "Explain what is a tardigrade.",
    max_tokens = 100
)

print(response.choices[0].text)